[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhisheksreesaila/mojo-gpu-tutorials/blob/main/013_blocklevel.ipynb)

In [ ]:
%%capture
!pip install mojo

# 🧱 Block-Level GPU Programming

### 🛶 But First... Let's Refresh: What Is a Warp?

- A **Warp** is the smallest unit of execution on a GPU.
- It's a group of **32 threads** 🧵 that are physically "married" at the hardware level to execute the exact same instruction at the exact same time ⏱️

<img src="../../assets/011_gpu_warp_speed.png" width="600" height="400">

**To explain it simply:**
Think of a warp like a **32-person rowing team** 🚣‍♂️ where everyone must move their oars in perfect lockstep 🔄

---

### 🧩 From Warps → Blocks

- Extending warp programming to **multiple warps** in a block = **block-level programming** 🏗️

- 🔧 Block-level programming provides **hardware-optimized building blocks** that coordinate hundreds of threads across multiple warps with a single function call.

- ⚠️ Manual block coordination (using shared memory and barriers) is **error-prone and verbose**. Mojo collapses this complexity into three fundamental patterns.

- 🚀 In short: enable **multi-warp programming** in a block!

### 🎁 What Does Mojo Provide?

| Pattern | Mojo Primitive | Goal | The "Old Way" |
| --- | --- | --- | --- |
| 📥 **All → One** | `block.sum()` | Collapse values into one result | 15+ lines of tree-reduction |
| 📊 **All → Each** | `block.prefix_sum()` | Give every thread a unique rank | Slow atomic operations |
| 📡 **One → All** | `block.broadcast()` | Share a parameter with everyone | Manual shared memory staging |

---

<img src="../../assets/013_block_pics.jpg" width="800" height="600">

---

## 1️⃣ `block.sum()` — Reduction 📥

**Every thread contributes → Thread 0 gets the total.**

Use this for totals, averages, and dot products. The hardware performs butterfly shuffles within each warp and uses high-speed shared memory to bridge the warps automatically.

### 🔍 API Breakdown

```mojo
// One line replaces manual tree-reduction loops
total = block.sum[block_size=128, broadcast=False](my_partial_product)

```

* **What Disappeared:** ❌ `barrier()` calls, ❌ shared memory management, ❌ stride-based indexing.

In [1]:
from IPython.display import display, HTML

# Create a sandboxed iframe with proper isolation
iframe_code = """
<iframe 
    sandbox="allow-scripts allow-same-origin"
    src="https://raw.githack.com/abhisheksreesaila/mojo-gpu-tutorials/main/visualize/gpu_block_sum.html"
    width="100%" 
    height="650px"
    style="border: 1px solid #ccc; border-radius: 5px; background: white;">
</iframe>
"""

display(HTML(iframe_code))


---

## 2️⃣ `block.prefix_sum()` — Scan 📊
## 📊 What is a Prefix Sum? (The Definition)

By definition, a **Prefix Sum** (also called a **Scan**) is an operation where each element in a sequence is replaced by the sum of all elements that came before it.


* **Input Array:** `[1, 1, 0, 1]`
* **Prefix Sum Result:** `[0, 1, 2, 2]` (Exclusive) <---- used and popular in GPU world!
* **Prefix Sum Result:** `[1, 2, 2, 3]` (Inclusive)

It is a simple "running total" that tells every thread exactly how many "matches" happened in the line ahead of them.

---

## 🚀 Practical Use Cases in the GPU World

While the math is simple, the **implications** for GPU hardware are massive. Here is how we actually use those numbers:

* **Assigning Memory Locations:** This is the #1 use case. The result of the sum acts as a **unique index**. It answers the question: *"I am the 4th matching thread, so I should write my data to `output[3]`."*
* **Stream Compaction (Packing):** It allows the GPU to take "scattered" data and pack it tightly at the front of a buffer. This makes the data much faster for the CPU or the next GPU kernel to read, as there are no "empty gaps" to skip.
* **Collision Prevention:** Because every thread receives a unique number from the sum, it is mathematically impossible for two threads to try to write to the same memory address. It replaces slow "atomic locks" with fast math.
* **Variable-Length Work:** If each thread is generating a different number of outputs (e.g., Thread A makes 2 items, Thread B makes 5), a Prefix Sum calculates exactly where Thread B’s items should start in the global list so they don't overwrite Thread A's.

---

## 🧱 The Workflow: Definition ⮕ Action

| Step | Action | The Math |
| --- | --- | --- |
| **1. Predicate** | **The Filter** | Identify the `1`s and `0`s (e.g., `WHERE val >= 10`). |
| **2. Prefix Sum** | **The Calculation** | **Definition:** Sum all prior elements to get a running total. |
| **3. Write** | **The Assignment** | Use that total as the **Memory Location** to save your data. |

---




In [ ]:
from IPython.display import display, HTML

# Create a sandboxed iframe with proper isolation
iframe_code = """
<iframe 
    sandbox="allow-scripts allow-same-origin"
    src="https://raw.githack.com/abhisheksreesaila/mojo-gpu-tutorials/main/visualize/gpu_block_prefix.html"
    width="100%" 
    height="650px"
    style="border: 1px solid #ccc; border-radius: 5px; background: white;">
</iframe>
"""

display(HTML(iframe_code))

---

## 3️⃣ `block.broadcast()` — Distribution 📡

**One thread has a value → All threads receive it.**

Used for sharing computed parameters, such as a calculated mean for normalization.

### 🏆 The "Normalize" Workflow

1. **📥 `block.sum()**`: All threads send values to Thread 0.
2. **Compute**: Thread 0 calculates `mean = total / size`.
3. **📡 `block.broadcast()**`: Thread 0 blasts the `mean` back to every thread.
4. **Parallel Action**: All threads normalize their data simultaneously.

---



In [ ]:
from IPython.display import display, HTML

# Create a sandboxed iframe with proper isolation
iframe_code = """
<iframe 
    sandbox="allow-scripts allow-same-origin"
    src="https://raw.githack.com/abhisheksreesaila/mojo-gpu-tutorials/main/visualize/gpu_block_broadcast.html"
    width="100%" 
    height="650px"
    style="border: 1px solid #ccc; border-radius: 5px; background: white;">
</iframe>
"""

display(HTML(iframe_code))


## 🏁 Key Takeaways

* **Hardware Managed:** Cross-warp coordination (the hardest part of GPU coding) is now invisible to you.
* **Composition:** Real algorithms are just chains of these three patterns (e.g., Sum → Compute → Broadcast).
* **Zero Atomics:** You get the performance of contiguous memory packing without the bottleneck of atomic locks.

In [3]:
import mojo.notebook

In [6]:
%%mojo

from gpu import thread_idx, block_idx, block_dim, grid_dim, barrier
from os.atomic import Atomic
from gpu.warp import WARP_SIZE
from gpu import block
from gpu.host import DeviceContext
from gpu.memory import AddressSpace
from layout import Layout, LayoutTensor
from sys import argv
from testing import assert_equal
from math import floor

#####################################################################################################################################################################

comptime SIZE = 128
comptime TPB = 128
comptime NUM_BINS = 2  # 0,1
comptime in_layout = Layout.row_major(SIZE)
comptime out_layout = Layout.row_major(1)
comptime dtype = DType.float32

# --- KERNEL 1: DOT PRODUCT ---
fn block_sum_dot_product[
    in_layout: Layout, out_layout: Layout, tpb: Int
](
    output: LayoutTensor[dtype, out_layout, MutAnyOrigin],
    a: LayoutTensor[dtype, in_layout, ImmutAnyOrigin],
    b: LayoutTensor[dtype, in_layout, ImmutAnyOrigin],
    size: Int,
):
    global_i = Int(block_dim.x * block_idx.x + thread_idx.x)
    local_i = thread_idx.x

    var partial_product: Scalar[dtype] = 0.0
    if global_i < size:
        partial_product = a[global_i][0] * b[global_i][0] # In Mojo, when you see input_data[global_i], you aren't just getting a single number; you are technically addressing a SIMD vector at that memory location.
                                                           # The Index: The [0] is you telling Mojo: "I know this memory location might be able to hold a whole pack of numbers, but I specifically want the first (0-th) element of the vector at this address

    total = block.sum[block_size=tpb, broadcast=False](
        val=SIMD[DType.float32, 1](partial_product)
    )

    if local_i == 0:
        output[0] = total[0]


# --- KERNEL 2: HISTOGRAM ---
comptime bin_layout = Layout.row_major(SIZE) 

fn block_histogram_bin_extract[
    in_layout: Layout, bin_layout: Layout, out_layout: Layout, tpb: Int
](
    input_data: LayoutTensor[dtype, in_layout, ImmutAnyOrigin],
    bin_output: LayoutTensor[dtype, bin_layout, MutAnyOrigin],
    count_output: LayoutTensor[DType.int32, out_layout, MutAnyOrigin],
    size: Int,
    target_bin: Int,
    num_bins: Int,
):
    global_i = Int(block_dim.x * block_idx.x + thread_idx.x)
    local_i = Int(thread_idx.x)

    # Every thread has its own private copy of these variables. 
    # There is NO risk of threads overwriting each other's 'my_value'.
    var my_value: Scalar[dtype] = 0.0
    var my_bin: Int = -1

    # 3. The "Load & Map" Phase (SELECT data_val, floor(...) as bin)
    if global_i < size:
        # We grab the 0-th element of the SIMD vector at our global index.
        my_value = input_data[global_i][0]
        
        # Calculate which "bucket" this value falls into. This could be anything. just picked a simple example
        my_bin = Int(floor(my_value * num_bins))  # 0.532 * 3 = 1.596 = floor(1.596) = 1 <<<< bin = 1
        
        # Safety Clamping: Ensure we don't calculate a bin index outside 0 to num_bins-1. 
        if my_bin >= num_bins: my_bin = num_bins - 1
        if my_bin < 0: my_bin = 0

    # 4. Create a indicator flag..treat 1 = true and 0 as false. 
    # condition chosen :  my_bin == target_bin (This could be anything. just picked a simple example)
    var belongs_to_target: Int = 0
    if global_i < size and my_bin == target_bin:
        belongs_to_target = 1

    # 5. The "Secret Sauce" 
    # block.prefix_sum asks every thread: "How many 1s came before you?"   <-------- functionality
    # Every thread gets back a unique 'write_offset' (its orange Seat Number).  <----------- practical use case
    # This prevents 128 threads from crashing into the same memory slot.  <----------- practical use case
    write_offset = block.prefix_sum[
        dtype = DType.int32, block_size=tpb, exclusive=True
    ](val=SIMD[DType.int32, 1](belongs_to_target))


    if belongs_to_target == 1:
        bin_output[Int(write_offset[0])] = my_value

    # The VERY LAST thread in the block is the only one who knows the total.
    if local_i == tpb - 1:
        # Its offset + its own match status = the total matches for the whole block.
        total_count = write_offset[0] + belongs_to_target
        # It writes that total to a single dedicated slot for the CPU to read.
        count_output[0] = total_count


# --- KERNEL 3: NORMALIZE ---
comptime vector_layout = Layout.row_major(SIZE)

fn block_normalize_vector[
    in_layout: Layout, out_layout: Layout, tpb: Int
](
    input_data: LayoutTensor[dtype, in_layout, ImmutAnyOrigin],
    output_data: LayoutTensor[dtype, out_layout, MutAnyOrigin],
    size: Int,
):
    # standard : get global and local thread index
    global_i = Int(block_dim.x * block_idx.x + thread_idx.x)
    local_i = thread_idx.x

    #When you write var my_value: Scalar[dtype], you aren't creating one variable that 128 threads share. 
    # You are telling the GPU: "Every single thread that runs this code must carve out its own private space in its own Registers to hold its own version of my_value."
    var my_value: Scalar[dtype] = 0.0
    if global_i < size:
        my_value = input_data[global_i][0]

    # compute sum for the whole block and store in thread 0 (block sum!) 
    total_sum = block.sum[block_size=tpb, broadcast=False](
        val=SIMD[DType.float32, 1](my_value)
    )
    
    # only thread 0, can calcualte teh mean, since its know the size and has the computed total value of all threads in a block
    var mean_value: Scalar[dtype] = 1.0
    if local_i == 0:
        if total_sum[0] > 0.0:
            mean_value = total_sum[0] / Float32(size)
    
    # broadcast!
    broadcasted_mean = block.broadcast[
        dtype = DType.float32, width=1, block_size=tpb
    ](val=SIMD[DType.float32, 1](mean_value), src_thread=UInt(0))

    # Each thread mormalizing its value
    if global_i < size:
        normalized_value = my_value / broadcasted_mean[0]
        output_data[global_i] = normalized_value


def main():
    with DeviceContext() as ctx:
        # ======================================================================
        # 1. EXECUTE DOT PRODUCT
        # ======================================================================
        print("--- RUNNING DOT PRODUCT ---")
        dot_out = ctx.enqueue_create_buffer[dtype](1)
        dot_a = ctx.enqueue_create_buffer[dtype](SIZE)
        dot_b = ctx.enqueue_create_buffer[dtype](SIZE)
        
        var dot_expected: Scalar[dtype] = 0.0
        with dot_a.map_to_host() as a_h, dot_b.map_to_host() as b_h:
            for i in range(SIZE):
                a_h[i] = i
                b_h[i] = 2 * i
                dot_expected += a_h[i] * b_h[i]

        comptime dot_kernel = block_sum_dot_product[in_layout, out_layout, TPB]
        ctx.enqueue_function_checked[dot_kernel, dot_kernel](
            LayoutTensor[dtype, out_layout, MutAnyOrigin](dot_out),
            LayoutTensor[dtype, in_layout, ImmutAnyOrigin](dot_a),
            LayoutTensor[dtype, in_layout, ImmutAnyOrigin](dot_b),
            SIZE,
            grid_dim=(1, 1), block_dim=(TPB, 1),
        )
        ctx.synchronize()
        with dot_out.map_to_host() as res:
            print("Dot Product Result:", res[0], "| Expected:", dot_expected)
        print()

        # ======================================================================
        # 2. EXECUTE HISTOGRAM (Running for 1 sample bin)
        # ======================================================================
        print("--- RUNNING HISTOGRAM (Bin 2) ---")
        hist_in = ctx.enqueue_create_buffer[dtype](SIZE)
        hist_bin_data = ctx.enqueue_create_buffer[dtype](SIZE)
        hist_count = ctx.enqueue_create_buffer[DType.int32](1)
        
        with hist_in.map_to_host() as h_in:
            for i in range(SIZE):
                h_in[i] = Float32(i % 80) / 100.0

        target_bin = 1
        comptime hist_kernel = block_histogram_bin_extract[in_layout, bin_layout, out_layout, TPB]
        ctx.enqueue_function_checked[hist_kernel, hist_kernel](
            LayoutTensor[dtype, in_layout, ImmutAnyOrigin](hist_in),
            LayoutTensor[dtype, bin_layout, MutAnyOrigin](hist_bin_data),
            LayoutTensor[DType.int32, out_layout, MutAnyOrigin](hist_count),
            SIZE, target_bin, NUM_BINS,
            grid_dim=(1, 1), block_dim=(TPB, 1),
        )
        ctx.synchronize()
        with hist_count.map_to_host() as c:
            print("Items in Bin", target_bin, ":", c[0])
        print()

        # ======================================================================
        # 3. EXECUTE NORMALIZATION
        # ======================================================================
        print("--- RUNNING NORMALIZATION ---")
        norm_in = ctx.enqueue_create_buffer[dtype](SIZE)
        norm_out = ctx.enqueue_create_buffer[dtype](SIZE)
        
        with norm_in.map_to_host() as n_in:
            for i in range(SIZE):
                n_in[i] = Float32((i % 8) + 1)

        comptime norm_kernel = block_normalize_vector[in_layout, vector_layout, TPB]
        ctx.enqueue_function_checked[norm_kernel, norm_kernel](
            LayoutTensor[dtype, in_layout, ImmutAnyOrigin](norm_in),
            LayoutTensor[dtype, vector_layout, MutAnyOrigin](norm_out),
            SIZE,
            grid_dim=(1, 1), block_dim=(TPB, 1),
        )
        ctx.synchronize()
        with norm_out.map_to_host() as n_res:
            var sum_check: Float32 = 0.0
            for i in range(SIZE): sum_check += n_res[i]
            print("Normalization Check: Mean of output is", sum_check / SIZE)
        print("--- ALL OPERATIONS COMPLETE ---")

--- RUNNING DOT PRODUCT ---
Dot Product Result: 1381760.0 | Expected: 1381760.0

--- RUNNING HISTOGRAM (Bin 2) ---
Items in Bin 1 : 30

--- RUNNING NORMALIZATION ---
Normalization Check: Mean of output is 1.0
--- ALL OPERATIONS COMPLETE ---

